# Exam Analysis — Parametric analysis of the penalty

This notebook analyses:
1. Evolution of each student's grade as a function of the penalty
2. Heatmap of grades by student and penalty
3. Pass-rate ladder as a function of the penalty
4. Exact threshold penalty per student

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#16213e',
    'axes.edgecolor': '#e94560', 'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0', 'xtick.color': '#e0e0e0',
    'ytick.color': '#e0e0e0', 'grid.color': '#2a2a4a', 'grid.alpha': 0.5,
    'font.family': 'sans-serif', 'font.size': 11,
})

STUDENT_COLORS = [
    '#e94560', '#53d8fb', '#ffc947', '#00e676', '#a29bfe', '#ff6b6b',
    '#55efc4', '#fd79a8', '#74b9ff', '#fab1a0', '#dfe6e9', '#636e72',
]

POINTS_CORRECT = 0.11
POINTS_WRONG = -0.05
PASS_THRESHOLD = 5.0

CSV_PATH = Path('.') / '134_36018173_ZE3IFC005200_MP5072_A-Examen 1a evaluación-cualificacións.csv'

def parse_answer(val):
    if isinstance(val, (int, float)):
        return 'correct' if val > 0 else ('wrong' if val < 0 else 'unanswered')
    s = str(val).strip().strip("'").strip('\u2018')
    if s in ('-', '', 'nan'): return 'unanswered'
    try:
        num = float(s.replace(',', '.'))
        return 'correct' if num > 0 else ('wrong' if num < 0 else 'unanswered')
    except ValueError: return 'unanswered'

def parse_score(val):
    if isinstance(val, (int, float)): return float(val)
    s = str(val).strip().strip("'").strip('\u2018')
    if s in ('-', '', 'nan'): return 0.0
    try: return float(s.replace(',', '.'))
    except ValueError: return 0.0

df = pd.read_csv(CSV_PATH)
df = df[df['Apelidos'].str.strip() != 'Media xeral'].dropna(subset=['Apelidos']).copy()
q_cols = [c for c in df.columns if c.startswith('P.')]
NUM_QUESTIONS = len(q_cols)
df['Alumno'] = df['Nome'].astype(str).str.strip() + ' ' + df['Apelidos'].astype(str).str.strip()
df['Nombre_Corto'] = df['Nome'].astype(str).str.strip()

# Desambiguar
for name in df['Nombre_Corto'].value_counts().index:
    mask = df['Nombre_Corto'] == name
    if mask.sum() > 1:
        df.loc[mask, 'Nombre_Corto'] = (
            df.loc[mask, 'Nome'].str.strip() + ' ' +
            df.loc[mask, 'Apelidos'].str.split().str[0]
        )

grade_col = [c for c in df.columns if 'ualificaci' in c][0]
df['Nota_Original'] = df[grade_col].apply(parse_score)

answer_types = pd.DataFrame(index=df.index)
for col in q_cols:
    answer_types[col] = df[col].apply(parse_answer)

df['n_correct'] = (answer_types[q_cols] == 'correct').sum(axis=1)
df['n_wrong'] = (answer_types[q_cols] == 'wrong').sum(axis=1)

n_students = len(df)
print(f'Datos cargados: {n_students} alumnos, {NUM_QUESTIONS} preguntas')

## Grade formula with variable penalty

The grade is calculated as:

$$\text{grade} = \frac{n_{\text{correct}} \times 0.11 + n_{\text{errors}} \times \text{penalty}}{0.11 \times 94} \times 10$$

Where `penalty` ranges from −0.05 (current) to 0.00 (no penalty).

In [ ]:
def calc_grade(n_correct, n_wrong, penalty):
    """Nota con fórmula directa."""
    raw = n_correct * POINTS_CORRECT + n_wrong * penalty
    max_possible = POINTS_CORRECT * NUM_QUESTIONS
    return max(0.0, (raw / max_possible) * 10.0)

def find_threshold_penalty(n_correct, n_wrong):
    """Penalización máxima (más negativa) con la que el alumno aprueba."""
    # nota >= 5.0 → raw >= max_possible * 0.5
    # n_correct * 0.11 + n_wrong * p >= 0.11 * 94 * 0.5
    needed_raw = POINTS_CORRECT * NUM_QUESTIONS * (PASS_THRESHOLD / 10.0)
    available = n_correct * POINTS_CORRECT
    deficit = needed_raw - available
    
    if deficit <= 0:
        # Aprueba incluso con mucha penalización → buscar límite
        if n_wrong == 0:
            return -999  # aprueba siempre
        return -deficit / n_wrong  # penalización máxima (negativa)
    else:
        if n_wrong == 0:
            return None  # no aprueba ni sin penalización
        threshold = -deficit / n_wrong
        if threshold > 0:
            return None  # imposible
        return threshold

## 1. Threshold penalty per student

In [ ]:
thresholds = []
for _, row in df.iterrows():
    thr = find_threshold_penalty(row['n_correct'], row['n_wrong'])
    grade_0 = calc_grade(row['n_correct'], row['n_wrong'], 0)
    thresholds.append({
        'Alumno': row['Alumno'],
        'Nombre': row['Nombre_Corto'],
        'Aciertos': row['n_correct'],
        'Errores': row['n_wrong'],
        'Nota Original': row['Nota_Original'],
        'Nota sin pen.': round(grade_0, 2),
        'Pen. umbral': f'{thr:.6f}' if thr is not None and thr > -999 else ('SIEMPRE APRUEBA' if thr == -999 else 'IMPOSIBLE'),
        'pen_value': thr if thr is not None else float('inf'),
    })

thr_df = pd.DataFrame(thresholds).sort_values('pen_value')
thr_df[['Alumno', 'Aciertos', 'Errores', 'Nota Original', 'Nota sin pen.', 'Pen. umbral']]

In [ ]:
already_pass = [t for t in thresholds if t['pen_value'] != float('inf') and t['pen_value'] <= POINTS_WRONG]
need_change = [t for t in thresholds if t['pen_value'] != float('inf') and t['pen_value'] > POINTS_WRONG]
impossible = [t for t in thresholds if t['pen_value'] == float('inf')]

print(f'✅ Ya aprueban con penalización actual ({POINTS_WRONG}): {len(already_pass)}')
for t in already_pass:
    print(f'   {t["Alumno"]}')

print(f'\n🔧 Aprobarían reduciendo penalización: {len(need_change)}')
for t in sorted(need_change, key=lambda x: x['pen_value']):
    print(f'   {t["Alumno"]} → necesita pen. ≥ {t["pen_value"]:.4f}')

if impossible:
    print(f'\n❌ No aprueban ni sin penalización: {len(impossible)}')
    for t in impossible:
        print(f'   {t["Alumno"]} (nota sin pen.: {t["Nota sin pen."]})')

if need_change:
    pen_all_possible = max(t['pen_value'] for t in need_change)
    n_with_it = len(already_pass) + len(need_change)
    print(f'\n💡 Con penalización = {pen_all_possible:.4f}, aprueban {n_with_it}/{n_students}')

## 2. Grade evolution by penalty (line chart)

In [ ]:
penalties = np.linspace(POINTS_WRONG, 0.0, 200)

grade_matrix = np.zeros((n_students, len(penalties)))
student_names = df['Nombre_Corto'].values

for j, pen in enumerate(penalties):
    for i, (_, row) in enumerate(df.iterrows()):
        grade_matrix[i, j] = calc_grade(row['n_correct'], row['n_wrong'], pen)

fig, ax = plt.subplots(figsize=(14, 7))
fig.suptitle('Evolución de la Nota por Alumno según Penalización',
             fontweight='bold', fontsize=15)

for i in range(n_students):
    ax.plot(penalties, grade_matrix[i, :], linewidth=2.5,
            color=STUDENT_COLORS[i % len(STUDENT_COLORS)],
            label=student_names[i], alpha=0.9)
    # Etiquetar al final
    ax.text(0.002, grade_matrix[i, -1], f' {student_names[i]}',
            fontsize=8, va='center', 
            color=STUDENT_COLORS[i % len(STUDENT_COLORS)], fontweight='bold')

ax.axhline(y=PASS_THRESHOLD, color='#00e676', linestyle='--', linewidth=2.5,
           alpha=0.8, label='Umbral aprobado (5.0)')
ax.axvline(x=POINTS_WRONG, color='#e94560', linestyle='--', linewidth=2, alpha=0.7,
           label=f'Penalización actual ({POINTS_WRONG})')

# Marcar puntos de cruce
for t in need_change:
    pen_v = t['pen_value']
    if -0.05 <= pen_v <= 0:
        ax.plot(pen_v, PASS_THRESHOLD, 'o', markersize=10, color='white',
                markeredgecolor='#ffc947', markeredgewidth=2, zorder=10)

ax.set_xlabel('Penalización por Respuesta Errónea', fontsize=12)
ax.set_ylabel('Nota (/10)', fontsize=12)
ax.set_xlim(POINTS_WRONG - 0.005, 0.015)
ax.set_ylim(0, 10.5)
ax.legend(fontsize=7, loc='upper left', ncol=2, fancybox=True, framealpha=0.3)
ax.grid(True, alpha=0.3)
ax.axhspan(PASS_THRESHOLD, 10.5, alpha=0.05, color='#00e676')

plt.tight_layout()
plt.show()

## 3. Heatmap: students × penalty

In [ ]:
pen_subset = np.arange(POINTS_WRONG, 0.005, 0.005)
heat_matrix = np.zeros((n_students, len(pen_subset)))

for j, pen in enumerate(pen_subset):
    for i, (_, row) in enumerate(df.iterrows()):
        heat_matrix[i, j] = calc_grade(row['n_correct'], row['n_wrong'], pen)

# Ordenar por nota original
sort_idx = np.argsort(heat_matrix[:, 0])
heat_sorted = heat_matrix[sort_idx]
names_sorted = student_names[sort_idx]

cmap = LinearSegmentedColormap.from_list('grade_cmap', [
    (0.0, '#8b0000'), (0.3, '#e94560'), (0.49, '#ff8c00'),
    (0.50, '#00e676'), (0.70, '#53d8fb'), (1.0, '#a29bfe'),
])

fig, ax = plt.subplots(figsize=(16, max(5, n_students * 0.5)))
fig.suptitle('Heatmap: Nota por Alumno y Penalización', fontweight='bold', fontsize=15)

im = ax.imshow(heat_sorted, aspect='auto', cmap=cmap, vmin=0, vmax=10,
               interpolation='nearest')

ax.set_yticks(range(n_students))
ax.set_yticklabels(names_sorted, fontsize=9)
ax.set_xticks(range(len(pen_subset)))
ax.set_xticklabels([f'{p:.3f}' for p in pen_subset], fontsize=7, rotation=45, ha='right')
ax.set_xlabel('Penalización por Error', fontsize=12)

for i in range(n_students):
    for j in range(len(pen_subset)):
        grade = heat_sorted[i, j]
        text_color = 'white' if grade < 4 or grade > 7 else 'black'
        ax.text(j, i, f'{grade:.1f}', ha='center', va='center',
                fontsize=7, fontweight='bold', color=text_color)

cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Nota', fontsize=11)
cbar.ax.axhline(y=PASS_THRESHOLD, color='white', linewidth=2)

plt.tight_layout()
plt.show()

## 4. Pass-rate ladder as a function of the penalty

In [ ]:
pen_fine = np.linspace(POINTS_WRONG, 0.0, 500)
n_pass_curve = []
for pen in pen_fine:
    count = sum(
        1 for _, row in df.iterrows()
        if calc_grade(row['n_correct'], row['n_wrong'], pen) >= PASS_THRESHOLD
    )
    n_pass_curve.append(count)

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Número de Aprobados en Función de la Penalización',
             fontweight='bold', fontsize=15)

ax.fill_between(pen_fine, n_pass_curve, alpha=0.3, color='#a29bfe')
ax.plot(pen_fine, n_pass_curve, linewidth=3, color='#a29bfe')

ax.axvline(x=POINTS_WRONG, color='#e94560', linestyle='--', linewidth=2, alpha=0.7,
           label=f'Actual ({POINTS_WRONG})')
ax.axhline(y=n_students, color='#00e676', linestyle=':', linewidth=1.5,
           alpha=0.5, label=f'Todos ({n_students})')

# Marcar cambios escalonados
prev = n_pass_curve[0]
for k, (pen, n_p) in enumerate(zip(pen_fine, n_pass_curve)):
    if n_p != prev:
        ax.plot(pen, n_p, 'o', markersize=10, color='white',
                markeredgecolor='#ffc947', markeredgewidth=2, zorder=10)
        ax.annotate(f'{n_p}/{n_students}\n(pen={pen:.4f})',
                    xy=(pen, n_p), xytext=(pen + 0.003, n_p + 0.3),
                    fontsize=8, fontweight='bold', color='#ffc947',
                    arrowprops=dict(arrowstyle='->', color='#ffc947', lw=1.5))
        prev = n_p

ax.set_xlabel('Penalización por Error', fontsize=12)
ax.set_ylabel('Nº de Aprobados', fontsize=12)
ax.set_xlim(POINTS_WRONG - 0.005, 0.015)
ax.set_ylim(0, n_students + 1)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Detailed parametric sweep

In [ ]:
pen_sweep = np.arange(POINTS_WRONG, 0.005, 0.005)
sweep_results = []
for pen in pen_sweep:
    grades_list = [calc_grade(row['n_correct'], row['n_wrong'], pen) for _, row in df.iterrows()]
    grades_arr = np.array(grades_list)
    n_p = np.sum(grades_arr >= PASS_THRESHOLD)
    sweep_results.append({
        'Penalización': round(pen, 3),
        'Aprobados': f'{n_p}/{n_students}',
        '% Aprobados': round(n_p / n_students * 100, 0),
        'Media': round(np.mean(grades_arr), 2),
        'Mín': round(np.min(grades_arr), 2),
        'Máx': round(np.max(grades_arr), 2),
    })

pd.DataFrame(sweep_results)

In [ ]:
# Gráfica doble: % aprobados y media vs penalización
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Barrido Paramétrico: Efecto de la Penalización por Error',
             fontweight='bold', fontsize=15)

pens = [r['Penalización'] for r in sweep_results]
passes = [r['% Aprobados'] for r in sweep_results]
means = [r['Media'] for r in sweep_results]

ax1.plot(pens, passes, '-o', color='#00e676', linewidth=2.5, markersize=5, alpha=0.9)
ax1.axhline(y=50, color='white', linestyle=':', linewidth=1, alpha=0.3)
ax1.axvline(x=POINTS_WRONG, color='#e94560', linestyle='--', linewidth=2,
            alpha=0.7, label=f'Actual ({POINTS_WRONG})')
ax1.set_xlabel('Penalización por Error')
ax1.set_ylabel('% Aprobados')
ax1.set_title('Tasa de Aprobados vs Penalización')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 105)

ax2.plot(pens, means, '-o', color='#ffc947', linewidth=2.5, markersize=5, alpha=0.9)
ax2.axhline(y=PASS_THRESHOLD, color='#00e676', linestyle='--', linewidth=2,
            alpha=0.7, label='Umbral (5.0)')
ax2.axvline(x=POINTS_WRONG, color='#e94560', linestyle='--', linewidth=2,
            alpha=0.7, label=f'Actual ({POINTS_WRONG})')
ax2.set_xlabel('Penalización por Error')
ax2.set_ylabel('Nota Media')
ax2.set_title('Nota Media vs Penalización')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusions of the penalty analysis

1. **Each student has a threshold penalty** above which they pass. This depends exclusively on their number of correct answers and errors.
2. **The line chart** visually shows where each student crosses the 5.0 threshold. Lines with steeper slopes correspond to students with more errors (more affected by the penalty).
3. **The heatmap** provides a compact view of all students and penalties simultaneously. The colour change red→green marks exactly the passing point.
4. **The pass-rate ladder** shows that the pass rate rises in steps: each step corresponds to a specific student crossing the threshold.
5. **Students who do not pass even without penalty** are those whose number of correct answers is insufficient (< 47/94 = 50% of questions). For these cases, only a combination of question removal + penalty reduction could be effective.
6. **Recommendation**: The optimal penalty depends on the pedagogical objective. If the goal is to discourage random guessing while maintaining some flexibility, an intermediate penalty (e.g. −0.02 to −0.03) may be a reasonable compromise.